# Analisi POI per Strutture Torino

Questo notebook analizza i Punti di Interesse (POI) attorno alle strutture nel comune di Torino. Per ogni struttura, determina i POI presenti nel raggio di 2 km per ogni categoria e amenity, e conta quanti ce ne sono nei raggi di 250 m, 500 m, 1 km e 2 km.

In [10]:
import pandas as pd
import json
import numpy as np
from sklearn.neighbors import BallTree
from sklearn.metrics.pairwise import haversine_distances
import os
import sys

# Aggiungi il percorso del progetto
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, project_root)

from app.data.loaders import load_and_merge_data
from amenities_config import CATEGORIES, CATEGORY_AMENITIES
from notebook_config import POIS_PATH

In [11]:
# Selezione use case
USE_CASE = 'studentati'  # Cambia in 'ospizi' per analizzare ospizi

from notebook_config import get_config, POIS_PATH

config = get_config(USE_CASE)
DATA_PATH = config['data_path']
OUTPUT_PATH = config['output_path']

In [12]:
# Caricamento dati strutture
strutture_path = DATA_PATH
strutture_df = pd.read_csv(strutture_path)
print(f"Caricati {len(strutture_df)} {USE_CASE}")
strutture_df.head()

Caricati 29 studentati


,Struttura,Indirizzo,Lat,Lon
0,Appartamenti ex MOI,"Via Giordano Bruno 201, 10134 Torino",45.030319,7.655493
1,Appartamenti Fondazione Saracco,"Via Giotto 51, 10126 Torino",45.042603,7.674042
2,Appartamenti Residenza Universitaria Farini,"Corso Farini 32, 10153 Torino",45.072933,7.701820
3,Appartamenti Turati,"Corso Filippo Turati 6, 10128 Torino",45.052169,7.669122
4,Appartamento Madama Cristina,"Corso Raffaello 20, 10128 Torino",45.051260,7.678723


In [13]:
# Caricamento dati POI
pois_path = POIS_PATH
with open(pois_path, 'r', encoding='utf-8') as f:
    pois_by_category = json.load(f)

print("Categorie POI caricate:", list(pois_by_category.keys()))

# Costruisci indice spaziale come in preprocess_immobili
def build_spatial_index(pois_by_category):
    all_pois = []
    for category, amenities in pois_by_category.items():
        for amenity, pois_list in amenities.items():
            for poi in pois_list:
                poi['category'] = category
                poi['amenity'] = amenity
                all_pois.append(poi)
    
    if not all_pois:
        return None, []
    
    coords_rad = np.radians([[p['lat'], p['lon']] for p in all_pois])
    tree = BallTree(coords_rad, metric='haversine')
    
    return tree, all_pois

tree, all_pois = build_spatial_index(pois_by_category)
print(f"Totale POI: {len(all_pois)}")

Categorie POI caricate: ['sanità', 'mobilità', 'verde', 'sport', 'commerciale', 'educazione']
Totale POI: 24230


In [14]:
# Definizione categorie e amenity
print("Categorie disponibili:")
for cat, amenities in CATEGORY_AMENITIES.items():
    print(f"- {cat}: {len(amenities)} amenity")

Categorie disponibili:
- sanità: 10 amenity
- mobilità: 13 amenity
- verde: 14 amenity
- sport: 12 amenity
- commerciale: 1 amenity
- educazione: 12 amenity


In [15]:
# Funzione per analizzare POI per una struttura
def analyze_pois_for_location(lat, lon, tree, all_pois, max_radius_km=2.0):
    """
    Per una location, trova tutti i POI entro max_radius_km e conta per raggi.
    """
    location_coords = np.radians([[lat, lon]])
    
    # Converti raggio in radianti (approssimazione per piccoli raggi)
    radii_km = [0.25, 0.5, 1.0, 2.0]
    radii_rad = [r / 6371.0 for r in radii_km]
    
    results = {}
    
    for category in CATEGORY_AMENITIES.keys():
        results[category] = {}
        for amenity_key, amenity_value in CATEGORY_AMENITIES[category]:
            amenity = amenity_value if amenity_value != '*' else amenity_key
            results[category][amenity] = {
                'pois_within_2km': [],
                'counts': {'250m': 0, '500m': 0, '1km': 0, '2km': 0}
            }
    
    # Trova tutti i POI entro 2km
    indices = tree.query_radius(location_coords, r=radii_rad[-1])[0]
    nearby_pois = [all_pois[i] for i in indices]
    
    # Calcola distanze esatte
    if nearby_pois:
        pois_coords = np.radians([[p['lat'], p['lon']] for p in nearby_pois])
        distances = haversine_distances(location_coords, pois_coords)[0] * 6371.0  # km
        
        for i, poi in enumerate(nearby_pois):
            dist = distances[i]
            cat = poi['category']
            amenity = poi['amenity']
            
            if cat in results and amenity in results[cat]:
                results[cat][amenity]['pois_within_2km'].append({
                    'id': poi['id'],
                    'lat': poi['lat'],
                    'lon': poi['lon'],
                    'distance_km': round(dist, 3)
                })
                
                # Conta nei raggi
                if dist <= 0.25:
                    results[cat][amenity]['counts']['250m'] += 1
                if dist <= 0.5:
                    results[cat][amenity]['counts']['500m'] += 1
                if dist <= 1.0:
                    results[cat][amenity]['counts']['1km'] += 1
                if dist <= 2.0:
                    results[cat][amenity]['counts']['2km'] += 1
    
    return results

In [16]:
# Analisi per tutte le strutture
results = []
for idx, row in strutture_df.iterrows():
    struttura = row['Struttura']
    lat = row['Lat']
    lon = row['Lon']
    
    print(f"Analizzando {struttura}...")
    analysis = analyze_pois_for_location(lat, lon, tree, all_pois)
    
    for category, amenities in analysis.items():
        for amenity, data in amenities.items():
            result_row = {
                'struttura': struttura,
                'lat': lat,
                'lon': lon,
                'categoria': category,
                'amenity': amenity,
                'count_250m': data['counts']['250m'],
                'count_500m': data['counts']['500m'],
                'count_1km': data['counts']['1km'],
                'count_2km': data['counts']['2km'],
                'pois_within_2km': ','.join([str(poi['id']) for poi in data['pois_within_2km']])  # ID separati da virgola
            }
            results.append(result_row)

results_df = pd.DataFrame(results)
print(f"Analisi completata. Totale righe: {len(results_df)}")

Analizzando Appartamenti ex MOI...
Analizzando Appartamenti Fondazione Saracco...
Analizzando Appartamenti Residenza Universitaria Farini...
Analizzando Appartamenti Turati...
Analizzando Appartamento Madama Cristina...
Analizzando Residenza universitaria Alexandra...
Analizzando Residenza universitaria Borsellino...
Analizzando Residenza universitaria Campus SanPaolo...
Analizzando Residenza universitaria Cappel Verde...
Analizzando Residenza universitaria Carlo Mollino...
Analizzando Residenza universitaria Cavour...
Analizzando Residenza universitaria Cercenasco (chiusa per lavori)...
Analizzando Residenza universitaria Codegone...
Analizzando Residenza universitaria CStudio...
Analizzando Residenza universitaria CXTURIN | MARCONI...
Analizzando Residenza universitaria CXTURIN | VANCHIGLIA...
Analizzando Residenza universitaria Duca...
Analizzando Residenza universitaria Giulia di Barolo...
Analizzando Residenza Universitaria Lingotto...
Analizzando Residenza universitaria Olimpia..

In [17]:
# Salvataggio del dataset
output_path = OUTPUT_PATH
results_df.to_csv(output_path, index=False)
print(f"Dataset salvato in {output_path}")

# Mostra un esempio
results_df.head()

Dataset salvato in analisi_poi_studentati_torino.csv


,struttura,lat,lon,categoria,amenity,count_250m,count_500m,count_1km,count_2km,pois_within_2km
0,Appartamenti ex MOI,45.030319,7.655493,sanità,healthcare,0,0,0,0,
1,Appartamenti ex MOI,45.030319,7.655493,sanità,pharmacy,0,2,12,50,"3452687789,4475300161,2592486338,3437215229,41..."
2,Appartamenti ex MOI,45.030319,7.655493,sanità,dentist,0,0,0,4,"9235444636,10245051371,9254763669,10751445003"
3,Appartamenti ex MOI,45.030319,7.655493,sanità,veterinary,0,0,2,4,"3147953784,8071508324,10698741095,8518082749"
4,Appartamenti ex MOI,45.030319,7.655493,sanità,optician,0,0,0,0,


In [18]:
# Esecuzione per entrambi i use case
def run_full_analysis(use_case):
    print(f"Elaborando {use_case}...")
    config = get_config(use_case)
    DATA_PATH = config['data_path']
    OUTPUT_PATH = config['output_path']
    
    strutture_df = pd.read_csv(DATA_PATH)
    print(f"Caricati {len(strutture_df)} {use_case}")
    
    results = []
    for idx, row in strutture_df.iterrows():
        struttura = row['Struttura']
        lat = row['Lat']
        lon = row['Lon']
        
        print(f"Analizzando {struttura}...")
        analysis = analyze_pois_for_location(lat, lon, tree, all_pois)
        
        for category, amenities in analysis.items():
            for amenity, data in amenities.items():
                result_row = {
                    'struttura': struttura,
                    'lat': lat,
                    'lon': lon,
                    'categoria': category,
                    'amenity': amenity,
                    'count_250m': data['counts']['250m'],
                    'count_500m': data['counts']['500m'],
                    'count_1km': data['counts']['1km'],
                    'count_2km': data['counts']['2km'],
                    'pois_within_2km': ','.join([str(poi['id']) for poi in data['pois_within_2km']])  # ID separati da virgola
                }
                results.append(result_row)
    
    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Analisi completata per {use_case}. Totale righe: {len(results_df)}. Salvato in {OUTPUT_PATH}")

# Esegui per entrambi i use case
for uc in ['studentati', 'ospizi']:
    run_full_analysis(uc)

Elaborando studentati...
Caricati 29 studentati
Analizzando Appartamenti ex MOI...
Analizzando Appartamenti Fondazione Saracco...
Analizzando Appartamenti Residenza Universitaria Farini...
Analizzando Appartamenti Turati...
Analizzando Appartamento Madama Cristina...
Analizzando Residenza universitaria Alexandra...
Analizzando Residenza universitaria Borsellino...
Analizzando Residenza universitaria Campus SanPaolo...
Analizzando Residenza universitaria Cappel Verde...
Analizzando Residenza universitaria Carlo Mollino...
Analizzando Residenza universitaria Cavour...
Analizzando Residenza universitaria Cercenasco (chiusa per lavori)...
Analizzando Residenza universitaria Codegone...
Analizzando Residenza universitaria CStudio...
Analizzando Residenza universitaria CXTURIN | MARCONI...
Analizzando Residenza universitaria CXTURIN | VANCHIGLIA...
Analizzando Residenza universitaria Duca...
Analizzando Residenza universitaria Giulia di Barolo...
Analizzando Residenza Universitaria Lingotto.